In [ ]:
# Cell 1: Imports and external-validation settings
from __future__ import annotations

import argparse
import gc
import hashlib
import itertools
import json
import math
import os
import re
import shutil
import sys
import tempfile
import time
import uuid
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import h5py
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.special import logsumexp
from scipy.optimize import minimize
from scipy.stats import gamma as scipy_gamma
import sklearn
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

class _EngineProxy:
    def __init__(self, prefix):
        object.__setattr__(self, "_prefix", prefix)
    def __getattr__(self, name):
        key = object.__getattribute__(self, "_prefix") + name
        if key not in globals():
            raise AttributeError(name)
        return globals()[key]
    def __setattr__(self, name, value):
        globals()[object.__getattribute__(self, "_prefix") + name] = value


In [ ]:
# Source cache utilities

source_RANDOM_SEED = 20260911

def source_stable_seed(*parts: Any) -> int:
    raw = "|".join(str(x) for x in parts).encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2**31 - 1)

source_FEATURE_ARRAY_KEYS = {
    "obs",
    "state_feat",
    "ch_anti",
    "clean",
}

def source_finite_model_array(x: np.ndarray) -> np.ndarray:
    a = np.asarray(x)
    if not np.issubdtype(a.dtype, np.number):
        return a
    if np.all(np.isfinite(a)):
        return a
    return np.nan_to_num(
        a,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
        copy=True,
    )

def source_assert_finite_model_inputs(
    data: Mapping[str, np.ndarray],
    context: str,
) -> None:
    bad = {}
    for key in source_FEATURE_ARRAY_KEYS:
        if key not in data:
            continue
        a = np.asarray(data[key])
        if np.issubdtype(a.dtype, np.number):
            n_bad = int(np.size(a) - np.sum(np.isfinite(a)))
            if n_bad:
                bad[key] = n_bad
    if bad:
        raise RuntimeError(
            f"{context}: non-finite model inputs remained after sanitization: {bad}"
        )

def source_load_cache(path: Path) -> Dict[str, Dict[str, np.ndarray]]:
    with np.load(path, allow_pickle=False) as z:
        out = {"train": {}, "test": {}}
        for key_name in z.files:
            if key_name.startswith("tr_"):
                split = "train"
                key = key_name[3:]
            elif key_name.startswith("te_"):
                split = "test"
                key = key_name[3:]
            else:
                continue

            arr = z[key_name]
            if key in source_FEATURE_ARRAY_KEYS:
                arr = source_finite_model_array(arr)
            out[split][key] = arr

    source_assert_finite_model_inputs(out["train"], f"{path.name}:train")
    source_assert_finite_model_inputs(out["test"], f"{path.name}:test")
    return out


In [ ]:
from __future__ import annotations

hb_GRID_DT = 0.5

hb_GRID_END = 20.0

hb_GRID = np.arange(0.0, hb_GRID_END + hb_GRID_DT / 2, hb_GRID_DT)

hb_MIN_GAP = 0.5

hb_MISS_PENALTY_SEC = 4.0

hb_CHANGE_WINDOWS = (3, 5, 9)

hb_RANDOM_SEED = 20260912

hb_PREDICTOR_KEYS = ('obs', 'state_feat')

def hb_stable_seed(*parts: Any) -> int:
    return int(hashlib.sha256('|'.join(map(str, parts)).encode()).hexdigest()[:8], 16) % (2 ** 31 - 1)

def hb_finite_model_array(x: np.ndarray) -> np.ndarray:
    return np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

def hb_predictor_view(data: Mapping[str, np.ndarray]) -> Dict[str, np.ndarray]:
    return {k: data[k] for k in hb_PREDICTOR_KEYS}

def hb_require_predictors_only(x: Mapping[str, np.ndarray]) -> None:
    if set(x) != set(hb_PREDICTOR_KEYS):
        raise RuntimeError(f'Prediction payload must contain ONLY {hb_PREDICTOR_KEYS}; got {sorted(x)}')

def hb_grouped_folds(pids: Sequence[str], seed: int, n: int=3) -> List[List[str]]:
    p = np.array(sorted(set(pids)), dtype=object)
    if len(p) < 3:
        raise RuntimeError('At least three source participants are required.')
    np.random.default_rng(seed).shuffle(p)
    return [q.tolist() for q in np.array_split(p, min(n, len(p))) if len(q)]

def hb_subset(data: Mapping[str, np.ndarray], mask: np.ndarray) -> Dict[str, np.ndarray]:
    return {k: np.asarray(v)[mask] for k, v in data.items()}

def hb_cat(parts: Sequence[Mapping[str, np.ndarray]]) -> Dict[str, np.ndarray]:
    return {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}

class hb_FitAudit:

    def __init__(self, allowed: Sequence[str], forbidden: Sequence[str], context: str):
        self.allowed = set(allowed)
        self.forbidden = set(forbidden)
        self.context = context
        self.rows = []

    def record(self, data: Mapping[str, np.ndarray], stage: str, held: Sequence[str]=()):
        actual = set(map(str, data['pid']))
        blocked = self.forbidden | set(held)
        if not actual or not actual.issubset(self.allowed) or actual & blocked:
            raise RuntimeError(f'{self.context}/{stage}: invalid fit membership {actual & blocked}')
        self.rows.append(dict(context=self.context, stage=stage, fit_participants=sorted(actual), forbidden=sorted(blocked), n_rows=int(len(data['pid']))))

def hb_smooth_series(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, float)
    if int(window) <= 1:
        return x
    w = min(int(window), len(x) if len(x) % 2 == 1 else len(x) - 1)
    if w < 3:
        return x
    if w % 2 == 0:
        w -= 1
    return signal.savgol_filter(x, w, 2 if w >= 5 else 1, mode='interp')

def hb_change_branch_features(data: Mapping[str, np.ndarray], frac: float) -> np.ndarray:
    """
    Compute multiscale change features for temporal-boundary estimation.

    Per scale:
      onset candidate
      peak candidate
      offset candidate
      max positive slope
      max negative slope
      peak amplitude
      curvature max / min
      local slope energy

    Plus cross-scale median/IQR/agreement summaries.
    """
    F = hb_finite_model_array(data['state_feat']).astype(float, copy=False)
    anti = F[:, :, 2]
    rows: List[List[float]] = []
    for a in anti:
        scale_rows = []
        for w in hb_CHANGE_WINDOWS:
            z = hb_smooth_series(a, int(w))
            dz = np.gradient(z, hb_GRID_DT)
            ddz = np.gradient(dz, hb_GRID_DT)
            early = np.flatnonzero(hb_GRID <= 8.0)
            oi = int(early[np.argmax(dz[early])]) if len(early) else int(np.argmax(dz))
            pi = int(oi + np.argmax(z[oi:]))
            mx = float(z[pi])
            aft = np.arange(pi + 1, len(z))
            offi = len(z) - 1
            if len(aft) and mx > 0:
                cand = aft[z[aft] <= float(frac) * mx]
                if len(cand):
                    offi = int(cand[0])
            feat = [float(hb_GRID[oi]), float(hb_GRID[pi]), float(hb_GRID[offi]), float(np.max(dz)), float(np.min(dz)), mx, float(np.max(ddz)), float(np.min(ddz)), float(np.mean(np.abs(dz)))]
            scale_rows.append(feat)
        A = np.asarray(scale_rows, float)
        flat = A.reshape(-1).tolist()
        for col in (0, 1, 2):
            x = A[:, col]
            flat.extend([float(np.median(x)), float(np.quantile(x, 0.75) - np.quantile(x, 0.25)), float(np.max(x) - np.min(x))])
        flat.extend([float(np.mean(A[:, 3])), float(np.mean(A[:, 4])), float(np.mean(A[:, 5])), float(np.std(A[:, 3])), float(np.std(A[:, 4]))])
        rows.append(flat)
    return hb_finite_model_array(np.asarray(rows, float))

def hb_enforce_geometry(on: np.ndarray, pk: np.ndarray, off: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    on = np.asarray(on, float).copy()
    pk = np.asarray(pk, float).copy()
    off = np.asarray(off, float).copy()
    on = np.clip(on, 0.0, hb_GRID_END - 2 * hb_MIN_GAP)
    pk = np.maximum(pk, on + hb_MIN_GAP)
    pk = np.clip(pk, hb_MIN_GAP, hb_GRID_END - hb_MIN_GAP)
    off = np.maximum(off, pk + hb_MIN_GAP)
    off = np.clip(off, 2 * hb_MIN_GAP, hb_GRID_END)
    return (on, pk, off)

def hb_balanced_accuracy(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, int)
    p = np.asarray(p, int)
    pos, neg = (y == 1, y == 0)
    sens = np.mean(p[pos] == 1) if np.any(pos) else np.nan
    spec = np.mean(p[neg] == 0) if np.any(neg) else np.nan
    return float(np.nanmean([sens, spec]))

def hb_trial_metric_summary(truth: Mapping[str, np.ndarray], pred: pd.DataFrame) -> Dict[str, float]:
    present = np.asarray(truth['present'], int) == 1
    cens = np.asarray(truth['censored'], int) == 1
    decision = pred['pred_response'].to_numpy(int)
    out: Dict[str, float] = {'response_balanced_accuracy': hb_balanced_accuracy(np.asarray(truth['present'], int), decision)}
    metrics = {}
    for b, tcol, pcol in [('onset', 'onset', 'pred_onset'), ('peak', 'peak', 'pred_peak'), ('offset', 'offset', 'pred_offset')]:
        t = np.asarray(truth[tcol], float)
        p = pred[pcol].to_numpy(float)
        valid_truth = present & np.isfinite(t)
        if b == 'offset':
            valid_truth &= ~cens
        err = np.full(np.sum(valid_truth), hb_MISS_PENALTY_SEC, float)
        idx = np.flatnonzero(valid_truth)
        finite = np.isfinite(p[idx]) & (decision[idx] == 1)
        err[finite] = np.abs(p[idx][finite] - t[idx][finite])
        metrics[f'{b}_mae_sec'] = float(np.mean(err)) if len(err) else np.nan
        metrics[f'{b}_localized_rate'] = float(np.mean(finite)) if len(err) else np.nan
    ton = np.asarray(truth['onset'], float)
    toff = np.asarray(truth['offset'], float)
    pon = pred['pred_onset'].to_numpy(float)
    poff = pred['pred_offset'].to_numpy(float)
    valid = present & ~cens & np.isfinite(ton) & np.isfinite(toff)
    idx = np.flatnonzero(valid)
    err = np.full(len(idx), hb_MISS_PENALTY_SEC, float)
    finite = np.isfinite(pon[idx]) & np.isfinite(poff[idx]) & (decision[idx] == 1)
    err[finite] = np.abs(poff[idx][finite] - pon[idx][finite] - (toff[idx][finite] - ton[idx][finite]))
    metrics['duration_mae_sec'] = float(np.mean(err)) if len(err) else np.nan
    out.update(metrics)
    scales = np.array([1.0, 1.0, 1.5, 1.5])
    vals = np.array([out['onset_mae_sec'], out['peak_mae_sec'], out['offset_mae_sec'], out['duration_mae_sec']])
    out['selection_loss'] = float(np.nanmean(vals / scales))
    return out

class hb_StableScaler:
    """Source-only, two-pass weighted standardization with safe constant columns.
    This avoids negative roundoff variances from weighted online variance updates.
    """

    def fit(self, X, sample_weight=None):
        X = np.asarray(X, float)
        if X.ndim != 2 or not np.isfinite(X).all():
            raise RuntimeError('Scaler needs finite 2D training features.')
        w = np.ones(len(X)) if sample_weight is None else np.asarray(sample_weight, float)
        if len(w) != len(X) or not np.isfinite(w).all() or (w < 0).any() or (w.sum() <= 0):
            raise RuntimeError('Invalid source sample weights.')
        self.mean_ = np.average(X, axis=0, weights=w)
        self.var_ = np.average((X - self.mean_) ** 2, axis=0, weights=w)
        self.var_ = np.maximum(self.var_, 0.0)
        eps = np.finfo(float).eps
        n = float(w.sum())
        constant = self.var_ <= n * eps * self.var_ + (n * self.mean_ * eps) ** 2
        self.scale_ = np.sqrt(self.var_)
        self.scale_[constant | (self.scale_ == 0)] = 1.0
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = np.asarray(X, float)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or (not np.isfinite(X).all()):
            raise RuntimeError('Scaler feature schema/finite check failed.')
        return (X - self.mean_) / self.scale_

def hb_fixed_order_path(p, stay_bias=0.0):
    """Compute an ordered five-state path from state probabilities."""
    logp = np.log(np.clip(p, 1e-09, 1.0))
    n, t, s = logp.shape
    dp = np.full((n, s), -np.inf)
    dp[:, 0] = logp[:, 0, 0]
    back = np.zeros((n, t, s), np.int8)
    for k in range(1, t):
        nxt = np.full_like(dp, -np.inf)
        for state in range(s):
            vs = dp[:, state] + stay_bias
            va = dp[:, state - 1] if state > 0 else np.full(n, -np.inf)
            keep = vs >= va
            nxt[:, state] = np.where(keep, vs, va) + logp[:, k, state]
            back[:, k, state] = np.where(keep, state, state - 1)
        dp = nxt
    states = np.full(n, s - 1, dtype=int)
    path = np.zeros((n, t), np.int8)
    ii = np.arange(n)
    for k in range(t - 1, -1, -1):
        path[:, k] = states
        if k > 0:
            states = back[ii, k, states]
    return path

def hb_hard_state_features(p, r):
    n, t, s = p.shape
    path = hb_fixed_order_path(p)
    occ = p.mean(axis=1)
    mx = p.max(axis=1)
    centres = (p * hb_GRID[None, :, None]).sum(axis=1) / np.maximum(p.sum(axis=1), 1e-12)
    summaries = np.stack((occ, mx, centres), axis=2).reshape(n, 15)
    dwell = np.column_stack([(path == state).mean(axis=1) for state in range(5)])
    oi = np.argmax(path == 1, axis=1)
    pi = np.argmax(np.where(path == 2, p[:, :, 2], -np.inf), axis=1)
    fi = np.argmax(path == 4, axis=1)
    ii = np.arange(n)
    times = np.column_stack((hb_GRID[oi], hb_GRID[pi], hb_GRID[fi]))
    conf = np.column_stack((p[ii, oi, 1], p[ii, pi, 2], p[ii, fi, 4]))
    ent = -(p * np.log(np.clip(p, 1e-09, 1))).sum(axis=2)
    return np.column_stack((summaries, dwell, times, conf, ent.mean(axis=1), ent.max(axis=1), r, (p[:, :, 1] + p[:, :, 2]).mean(axis=1), (p[:, :, 3] + p[:, :, 4]).mean(axis=1)))


In [ ]:
from __future__ import annotations

def hb_local_profiles(x, probabilities, anchors, response_prob, radius):
    hb_require_predictors_only(x)
    F = hb_finite_model_array(x['state_feat'])
    P = np.asarray(probabilities, float)
    n = len(F)
    rel = np.linspace(-float(radius), float(radius), 9)
    out = []
    for b in range(3):
        rows = []
        for i in range(n):
            centre = float(anchors[i, b])
            t = centre + rel
            f = np.column_stack([np.interp(t, hb_GRID, F[i, :, c]) for c in range(F.shape[2])])
            p = np.column_stack([np.interp(t, hb_GRID, P[i, :, c]) for c in range(P.shape[2])])
            seen = ((t >= 0) & (t <= hb_GRID_END)).astype(float)
            context = np.r_[anchors[i], response_prob[i], anchors[i, 1] - anchors[i, 0], anchors[i, 2] - anchors[i, 1]]
            rows.append(np.r_[f.ravel(), p.ravel(), seen, context])
        out.append(np.asarray(rows, float))
    return out

@dataclass
class hb_NoiseProfile:
    n_channels: int
    hbo_scale: np.ndarray
    hbr_scale: np.ndarray
    phi_hbo: float
    phi_hbr: float

def hb_safe_lag1(a: np.ndarray) -> float:
    vals = []
    for row in a:
        x, y = (row[:-1], row[1:])
        if len(x) >= 3 and np.std(x) > 1e-10 and (np.std(y) > 1e-10):
            vals.append(float(np.corrcoef(x, y)[0, 1]))
    return float(np.clip(np.nanmedian(vals) if vals else 0.5, 0.0, 0.95))

def hb_ar_noise(scale: np.ndarray, phi: float, rng: np.random.Generator, n_time: int) -> np.ndarray:
    C = len(scale)
    x = np.zeros((C, n_time), float)
    common = rng.normal(size=n_time)
    for k in range(n_time):
        eps = 0.75 * rng.normal(size=C) + 0.25 * common[k]
        if k == 0:
            x[:, k] = eps
        else:
            x[:, k] = phi * x[:, k - 1] + math.sqrt(max(1.0 - phi ** 2, 1e-06)) * eps
    return x * scale[:, None]

def hb_piecewise_shape(onset: float, rise: float, active: float, decay: float) -> Tuple[np.ndarray, float, float, np.ndarray]:
    peak = onset + rise
    active_end = peak + active
    offset = active_end + decay
    y = np.zeros(len(hb_GRID), float)
    states = np.zeros(len(hb_GRID), int)
    for i, tt in enumerate(hb_GRID):
        if tt < onset:
            y[i] = 0.0
            states[i] = 0
        elif tt < peak:
            u = (tt - onset) / max(rise, 1e-08)
            y[i] = 0.5 - 0.5 * np.cos(np.pi * u)
            states[i] = 1
        elif tt < active_end:
            y[i] = 1.0
            states[i] = 2
        elif tt < offset:
            u = (tt - active_end) / max(decay, 1e-08)
            y[i] = 0.5 + 0.5 * np.cos(np.pi * u)
            states[i] = 3
        else:
            y[i] = 0.0
            states[i] = 4
    return (y, peak, offset, states)

def hb_synthetic_trial(profile: hb_NoiseProfile, rng: np.random.Generator, positive: bool) -> Dict[str, Any]:
    H = hb_ar_noise(profile.hbo_scale, profile.phi_hbo, rng, len(hb_FULL_T))
    R = hb_ar_noise(profile.hbr_scale, profile.phi_hbr, rng, len(hb_FULL_T))
    systemic = rng.uniform(0.0, 0.8) * np.nanmedian(profile.hbo_scale) * np.sin(2 * np.pi * rng.uniform(0.06, 0.12) * hb_FULL_T + rng.uniform(0, 2 * np.pi))
    H += systemic[None, :]
    R += rng.uniform(0.5, 1.0) * systemic[None, :]
    if not positive:
        return dict(hbo=H, hbr=R, present=0, onset=np.nan, peak=np.nan, offset=np.nan, censored=0, clean_shape=np.zeros(len(hb_GRID), float), states=np.zeros(len(hb_GRID), int))
    onset = float(rng.uniform(0.5, 4.5))
    rise = float(rng.uniform(1.0, 3.5))
    active = float(rng.uniform(0.5, 3.0))
    if rng.random() < 0.18:
        decay = float(rng.uniform(7.0, 12.0))
    else:
        decay = float(rng.uniform(2.0, 8.0))
    shape, peak, offset, states = hb_piecewise_shape(onset, rise, active, decay)
    post_shape = np.zeros(len(hb_FULL_T), float)
    post_shape[hb_POST_MASK] = shape
    support = rng.uniform(0.55, 1.0, size=profile.n_channels)
    amp = rng.uniform(1.5, 4.5) * profile.hbo_scale
    H += (amp * support)[:, None] * post_shape[None, :]
    R -= (amp * rng.uniform(0.35, 0.7) * rng.uniform(0.7, 1.1, size=profile.n_channels))[:, None] * post_shape[None, :]
    return dict(hbo=H, hbr=R, present=1, onset=onset, peak=peak, offset=offset, censored=int(offset > hb_GRID_END - 0.25), clean_shape=shape, states=states)

def hb_baseline_z(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, float)
    bm = hb_FULL_T < 0
    med = np.nanmedian(arr[:, bm], axis=1, keepdims=True)
    mad = 1.4826 * np.nanmedian(np.abs(arr[:, bm] - med), axis=1, keepdims=True)
    sd = np.nanstd(arr[:, bm], axis=1, keepdims=True)
    scale = np.where(np.isfinite(mad) & (mad > 1e-08), mad, np.where(sd > 1e-08, sd, 1.0))
    return (arr - med) / scale

def hb_evidence(hbo: np.ndarray, hbr: np.ndarray) -> Dict[str, np.ndarray]:
    H = hb_baseline_z(hbo)[:, hb_POST_MASK]
    R = hb_baseline_z(hbr)[:, hb_POST_MASK]
    ghbo = np.nanmedian(H, axis=0)
    ghbr = np.nanmedian(R, axis=0)
    anti = 0.5 * (ghbo - ghbr)
    dhbo = np.gradient(ghbo, hb_GRID_DT)
    dhbr = np.gradient(ghbr, hb_GRID_DT)
    danti = np.gradient(anti, hb_GRID_DT)
    dH = np.gradient(H, hb_GRID_DT, axis=1)
    dR = np.gradient(R, hb_GRID_DT, axis=1)
    cons_hbo = np.mean(np.sign(dH) == np.sign(dhbo)[None, :], axis=0)
    cons_hbr = np.mean(np.sign(dR) == np.sign(dhbr)[None, :], axis=0)
    iqr_hbo = np.nanquantile(H, 0.75, axis=0) - np.nanquantile(H, 0.25, axis=0)
    iqr_hbr = np.nanquantile(R, 0.75, axis=0) - np.nanquantile(R, 0.25, axis=0)
    return dict(H=H, R=R, ghbo=ghbo, ghbr=ghbr, anti=anti, dhbo=dhbo, dhbr=dhbr, danti=danti, cons_hbo=cons_hbo, cons_hbr=cons_hbr, iqr_hbo=iqr_hbo, iqr_hbr=iqr_hbr)

def hb_whole_vector(ev: Mapping[str, np.ndarray]) -> np.ndarray:
    return np.concatenate([ev['ghbo'], ev['ghbr'], ev['anti'], ev['danti']]).astype(np.float32)


In [ ]:
from __future__ import annotations

def hb_state_features(ev: Mapping[str, np.ndarray]) -> np.ndarray:
    return np.column_stack([ev['ghbo'], ev['ghbr'], ev['anti'], ev['dhbo'], ev['dhbr'], ev['danti'], ev['cons_hbo'], ev['cons_hbr'], 0.5 * (ev['iqr_hbo'] + ev['iqr_hbr'])]).astype(np.float32)

hb_BASELINE = (-5.0, 0.0)

hb_FULL_T = np.arange(hb_BASELINE[0], hb_GRID_END + hb_GRID_DT / 2.0, hb_GRID_DT)

hb_POST_MASK = hb_FULL_T >= 0

hb_EXTERNAL_TEST_POS = 24

hb_EXTERNAL_TEST_NEG = 12

hb_BALINT_DEFAULT = Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/Balint/processed/balint_epochs_10hz.h5')

hb_KHAN_DEFAULT = Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/Khan/processed/khan2026_finger_epochs_10hz.h5')

def hb__decode_str_array(a):
    a = np.asarray(a)
    out = []
    for x in a.reshape(-1):
        if isinstance(x, bytes):
            out.append(x.decode('utf-8', errors='replace'))
        else:
            out.append(str(x))
    return np.asarray(out, dtype=str)

def hb__h5_dataset_inventory(path):
    rows = []
    with h5py.File(path, 'r') as h5:

        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows.append(dict(path='/' + name, shape=tuple(obj.shape), dtype=str(obj.dtype), ndim=int(obj.ndim)))
        h5.visititems(visit)
    return pd.DataFrame(rows)

def hb__all_datasets(h5):
    out = {}

    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            out['/' + name] = obj
    h5.visititems(visit)
    return out

def hb__score_name(name, tokens):
    low = name.lower()
    return sum((5 if low.endswith('/' + t) or low.split('/')[-1] == t else 1 for t in tokens if t in low))

def hb__find_time_dataset(datasets):
    candidates = []
    for name, ds in datasets.items():
        if ds.ndim != 1 or not np.issubdtype(ds.dtype, np.number):
            continue
        if len(ds) < 20 or len(ds) > 5000:
            continue
        score = hb__score_name(name, ('time', 'times', 'epoch_time', 't'))
        if score <= 0:
            continue
        a = np.asarray(ds[:], float).reshape(-1)
        if not np.isfinite(a).all() or not np.all(np.diff(a) > 0):
            continue
        if np.nanmin(a) > -0.1 or np.nanmax(a) < 10.0:
            continue
        candidates.append((score, name, a))
    if not candidates:
        raise RuntimeError('Could not identify a monotonic HDF5 epoch time vector spanning pre/post stimulus.')
    candidates.sort(key=lambda x: (-x[0], len(x[2]), x[1]))
    return (candidates[0][1], candidates[0][2])

def hb__find_participant_dataset(datasets):
    candidates = []
    allowed_leafs = {'participant', 'participants', 'participant_id', 'participant_ids', 'subject', 'subjects', 'subject_id', 'subject_ids', 'sub', 'sub_id'}
    for name, ds in datasets.items():
        if ds.ndim != 1:
            continue
        leaf = name.rstrip('/').split('/')[-1].lower()
        if leaf not in allowed_leafs:
            continue
        try:
            vals = hb__decode_str_array(ds[:])
        except Exception:
            continue
        unique = sorted(set(vals.tolist()))
        if not 2 <= len(unique) <= 200:
            continue
        looks_like = sum((bool(re.fullmatch('(?i)(?:sub[-_]?)?[A-Z]*\\d{1,4}', v.strip())) for v in unique))
        if looks_like == 0 and leaf in {'sub', 'sub_id'}:
            continue
        score = 10
        if 'participant' in leaf:
            score += 3
        if leaf.endswith('_id') or leaf.endswith('_ids'):
            score += 1
        candidates.append((score, name, vals))
    if not candidates:
        return (None, None)
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return (candidates[0][1], candidates[0][2])

def hb__normalize_trial_channel_time(a, n_trials, n_time, label):
    x = np.asarray(a, float)
    if x.ndim != 3:
        raise RuntimeError(f'{label}: expected 3D trial/channel/time data, got {x.shape}')
    t_axes = [i for i, s in enumerate(x.shape) if s == n_time]
    if len(t_axes) != 1:
        raise RuntimeError(f'{label}: could not uniquely identify time axis length {n_time} in shape {x.shape}')
    x = np.moveaxis(x, t_axes[0], -1)
    first_two = x.shape[:2]
    tr_axes = [i for i, s in enumerate(first_two) if s == n_trials]
    if len(tr_axes) != 1:
        raise RuntimeError(f'{label}: could not uniquely identify trial axis length {n_trials} in shape {x.shape}')
    if tr_axes[0] == 1:
        x = np.swapaxes(x, 0, 1)
    if x.shape[0] != n_trials or x.shape[-1] != n_time:
        raise RuntimeError(f'{label}: normalization failed, shape={x.shape}')
    return x

def hb__find_separate_chromophore_pair(datasets, n_trials, n_time):
    hbo = []
    hbr = []
    for name, ds in datasets.items():
        if ds.ndim != 3 or not np.issubdtype(ds.dtype, np.number):
            continue
        low = name.lower()
        score_o = hb__score_name(name, ('hbo', 'oxy', 'oxygenated'))
        score_r = hb__score_name(name, ('hbr', 'deoxy', 'deoxygenated'))
        if 'deoxy' in low or 'deoxygenated' in low:
            score_o = 0
        if score_o > 0:
            hbo.append((score_o, name, ds))
        if score_r > 0:
            hbr.append((score_r, name, ds))
    pairs = []
    for so, no, dso in hbo:
        for sr, nr, dsr in hbr:
            try:
                O = hb__normalize_trial_channel_time(dso[:], n_trials, n_time, no)
                R = hb__normalize_trial_channel_time(dsr[:], n_trials, n_time, nr)
            except RuntimeError:
                continue
            if O.shape == R.shape:
                pairs.append((so + sr, no, nr, O, R))
    if not pairs:
        return None
    pairs.sort(key=lambda x: (-x[0], x[1], x[2]))
    _, no, nr, O, R = pairs[0]
    return (no, nr, O, R)

def hb__find_channel_names(datasets):
    candidates = []
    for name, ds in datasets.items():
        if ds.ndim != 1:
            continue
        score = hb__score_name(name, ('channel_names', 'channels', 'ch_names', 'channel'))
        if score <= 0:
            continue
        try:
            vals = hb__decode_str_array(ds[:])
        except Exception:
            continue
        if len(vals) >= 2:
            candidates.append((score, name, vals))
    if not candidates:
        return (None, None)
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return (candidates[0][1], candidates[0][2])

def hb__normalize_combined_epochs(a, n_trials, n_time, n_channels, label):
    x = np.asarray(a, float)
    if x.ndim != 3:
        raise RuntimeError(f'{label}: combined epoch dataset must be 3D.')
    axes = list(range(3))
    ta = [i for i, s in enumerate(x.shape) if s == n_time]
    ca = [i for i, s in enumerate(x.shape) if s == n_channels]
    ra = [i for i, s in enumerate(x.shape) if s == n_trials]
    if len(ta) != 1 or len(ca) != 1 or len(ra) != 1:
        raise RuntimeError(f'{label}: could not uniquely map trial/channel/time axes in shape {x.shape}')
    x = np.transpose(x, (ra[0], ca[0], ta[0]))
    return x

def hb__split_combined_by_channel_names(datasets, n_trials, n_time):
    cn_path, names = hb__find_channel_names(datasets)
    if names is None:
        return None
    idx_o = []
    idx_r = []
    keys_o = []
    keys_r = []
    for i, name in enumerate(names):
        low = name.lower()
        is_r = 'hbr' in low or 'deoxy' in low or 'deoxygenated' in low
        is_o = 'hbo' in low or (('oxy' in low or 'oxygenated' in low) and (not is_r))
        key = re.sub('(?i)(hbo|hbr|deoxy(?:genated)?|oxy(?:genated)?)', '', low)
        key = re.sub('[^a-z0-9]+', '', key)
        if is_o:
            idx_o.append(i)
            keys_o.append(key)
        elif is_r:
            idx_r.append(i)
            keys_r.append(key)
    if not idx_o or not idx_r:
        return None
    map_o = {k: i for k, i in zip(keys_o, idx_o) if k}
    map_r = {k: i for k, i in zip(keys_r, idx_r) if k}
    common = sorted(set(map_o) & set(map_r))
    if common:
        io = np.asarray([map_o[k] for k in common], int)
        ir = np.asarray([map_r[k] for k in common], int)
    else:
        c = min(len(idx_o), len(idx_r))
        io = np.asarray(idx_o[:c], int)
        ir = np.asarray(idx_r[:c], int)
    candidates = []
    for path, ds in datasets.items():
        if ds.ndim != 3 or not np.issubdtype(ds.dtype, np.number):
            continue
        score = hb__score_name(path, ('epoch', 'epochs', 'data', 'hb', 'nirs'))
        if score <= 0:
            continue
        try:
            X = hb__normalize_combined_epochs(ds[:], n_trials=n_trials, n_time=n_time, n_channels=len(names), label=path)
        except RuntimeError:
            continue
        O = X[:, io, :]
        R = X[:, ir, :]
        candidates.append((score, path, cn_path, O, R))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))
    _, data_path, _, O, R = candidates[0]
    return (data_path + '::HbO/HbR channel split', cn_path, O, R)

def hb__load_flat_hdf5(path):
    with h5py.File(path, 'r') as h5:
        datasets = hb__all_datasets(h5)
        time_path, times = hb__find_time_dataset(datasets)
        participant_path, participants = hb__find_participant_dataset(datasets)
        if participants is None:
            return None
        n_trials = len(participants)
        n_time = len(times)
        pair = hb__find_separate_chromophore_pair(datasets, n_trials=n_trials, n_time=n_time)
        if pair is None:
            pair = hb__split_combined_by_channel_names(datasets, n_trials=n_trials, n_time=n_time)
        if pair is None:
            return None
        hbo_path, hbr_path, H, R = pair
    return dict(times=times, participant=participants, hbo=H, hbr=R, schema='FLAT_HDF5', audit=dict(time_dataset=time_path, participant_dataset=participant_path, hbo_source=hbo_path, hbr_source=hbr_path))

def hb__participant_group_candidates(h5):
    candidates = []

    def visit(name, obj):
        if not isinstance(obj, h5py.Group):
            return
        leaf = name.rstrip('/').split('/')[-1]
        if re.fullmatch('(?i)(?:sub[-_])?[CS]?\\d{1,3}', leaf):
            candidates.append(name)
    h5.visititems(visit)
    return sorted(set(candidates))

def hb__load_grouped_hdf5(path):
    participant_blocks = []
    audit_rows = []
    with h5py.File(path, 'r') as h5:
        global_ds = hb__all_datasets(h5)
        try:
            global_time_path, global_times = hb__find_time_dataset(global_ds)
        except RuntimeError:
            global_time_path, global_times = (None, None)
        groups = hb__participant_group_candidates(h5)
        for gp in groups:
            grp = h5[gp]
            datasets = {}

            def visit(name, obj):
                if isinstance(obj, h5py.Dataset):
                    datasets['/' + gp + '/' + name] = obj
            grp.visititems(visit)
            if not datasets:
                continue
            if global_times is not None:
                times = global_times
                time_path = global_time_path
            else:
                try:
                    time_path, times = hb__find_time_dataset(datasets)
                except RuntimeError:
                    continue
            hbo_ds = []
            hbr_ds = []
            for name, ds in datasets.items():
                if ds.ndim != 3 or not np.issubdtype(ds.dtype, np.number):
                    continue
                low = name.lower()
                so = hb__score_name(name, ('hbo', 'oxy', 'oxygenated'))
                sr = hb__score_name(name, ('hbr', 'deoxy', 'deoxygenated'))
                if 'deoxy' in low or 'deoxygenated' in low:
                    so = 0
                if so > 0:
                    hbo_ds.append((so, name, ds))
                if sr > 0:
                    hbr_ds.append((sr, name, ds))
            best = None
            for so, no, dso in hbo_ds:
                for sr, nr, dsr in hbr_ds:
                    if tuple(dso.shape) != tuple(dsr.shape):
                        continue
                    shape = dso.shape
                    t_axes = [i for i, s in enumerate(shape) if s == len(times)]
                    if len(t_axes) != 1:
                        continue
                    ta = t_axes[0]
                    other = [i for i in range(3) if i != ta]
                    trial_lengths = []
                    for n2, ds2 in datasets.items():
                        if ds2.ndim == 1 and hb__score_name(n2, ('trial', 'trials', 'label', 'event')) > 0:
                            trial_lengths.append(len(ds2))
                    trial_axis = None
                    for length in trial_lengths:
                        matching = [ax for ax in other if shape[ax] == length]
                        if len(matching) == 1:
                            trial_axis = matching[0]
                            break
                    if trial_axis is None:
                        channel_like = [ax for ax in other if shape[ax] <= 128]
                        if len(channel_like) == 1:
                            trial_axis = [ax for ax in other if ax != channel_like[0]][0]
                        else:
                            trial_axis = other[0]
                    channel_axis = [ax for ax in other if ax != trial_axis][0]
                    O = np.transpose(np.asarray(dso[:], float), (trial_axis, channel_axis, ta))
                    R = np.transpose(np.asarray(dsr[:], float), (trial_axis, channel_axis, ta))
                    score = so + sr
                    if best is None or score > best[0]:
                        best = (score, no, nr, O, R)
            if best is None:
                continue
            _, no, nr, O, R = best
            pid = gp.rstrip('/').split('/')[-1]
            participant_blocks.append((pid, times, O, R))
            audit_rows.append(dict(participant=pid, group='/' + gp, hbo_source=no, hbr_source=nr, time_source=time_path, n_trials=int(len(O)), n_channels=int(O.shape[1])))
    if not participant_blocks:
        return None
    source_t = participant_blocks[0][1]
    for pid, t, _, _ in participant_blocks:
        if len(t) != len(source_t) or not np.allclose(t, source_t, atol=1e-08, rtol=0):
            raise RuntimeError(f'Grouped HDF5 participant {pid} has a different time vector.')
    participants = []
    H = []
    R = []
    min_c = min((min(x[2].shape[1], x[3].shape[1]) for x in participant_blocks))
    for pid, _, h, r in participant_blocks:
        participants.extend([pid] * len(h))
        H.append(h[:, :min_c, :])
        R.append(r[:, :min_c, :])
    return dict(times=np.asarray(source_t, float), participant=np.asarray(participants, dtype=str), hbo=np.concatenate(H, axis=0), hbr=np.concatenate(R, axis=0), schema='GROUPED_HDF5', audit=dict(group_rows=audit_rows))

def hb_load_external_hdf5(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    flat = hb__load_flat_hdf5(path)
    if flat is not None:
        return flat
    grouped = hb__load_grouped_hdf5(path)
    if grouped is not None:
        return grouped
    inventory = hb__h5_dataset_inventory(path)
    raise RuntimeError('Could not resolve HbO/HbR/time/participant arrays in external HDF5.\nNo results were written because the HDF5 schema could not be resolved.\nHDF5 inventory:\n' + inventory.to_string(index=False))

def hb__participant_arrays(store, pid):
    mask = store['participant'].astype(str) == str(pid)
    if not np.any(mask):
        raise RuntimeError(f'External participant {pid} has no epochs.')
    H = np.asarray(store['hbo'][mask], float)
    R = np.asarray(store['hbr'][mask], float)
    t = np.asarray(store['times'], float)
    C = min(H.shape[1], R.shape[1])
    H = H[:, :C, :]
    R = R[:, :C, :]
    bm = (t >= hb_BASELINE[0]) & (t < hb_BASELINE[1])
    if bm.sum() < 4:
        raise RuntimeError(f'{pid}: external store has insufficient -5..0 s baseline.')
    finite_fraction = np.mean(np.isfinite(H[:, :, bm]) & np.isfinite(R[:, :, bm]), axis=(0, 2))
    hsd = np.nanmedian(np.nanstd(H[:, :, bm], axis=2), axis=0)
    rsd = np.nanmedian(np.nanstd(R[:, :, bm], axis=2), axis=0)
    valid = (finite_fraction >= 0.8) & np.isfinite(hsd) & np.isfinite(rsd) & (hsd > 1e-12) & (rsd > 1e-12)
    if valid.sum() < 2:
        valid = finite_fraction >= 0.8
    if valid.sum() < 2:
        raise RuntimeError(f'{pid}: fewer than two usable HbO/HbR channels.')
    return (t, H[:, valid, :], R[:, valid, :], valid)

def hb_noise_profile_from_external(store, pid):
    t, H, R, valid = hb__participant_arrays(store, pid)
    bm = (t >= hb_BASELINE[0]) & (t < hb_BASELINE[1])
    Hb = H[:, :, bm]
    Rb = R[:, :, bm]
    Hb = Hb - np.nanmedian(Hb, axis=2, keepdims=True)
    Rb = Rb - np.nanmedian(Rb, axis=2, keepdims=True)
    hs = np.nanmedian(np.nanstd(Hb, axis=2), axis=0)
    rs = np.nanmedian(np.nanstd(Rb, axis=2), axis=0)
    hs_positive = hs[np.isfinite(hs) & (hs > 0)]
    rs_positive = rs[np.isfinite(rs) & (rs > 0)]
    h_fallback = float(np.nanmedian(hs_positive)) if len(hs_positive) else 1.0
    r_fallback = float(np.nanmedian(rs_positive)) if len(rs_positive) else 1.0
    hs = np.where(np.isfinite(hs) & (hs > 1e-12), hs, h_fallback)
    rs = np.where(np.isfinite(rs) & (rs > 1e-12), rs, r_fallback)
    sample_H = Hb[:min(24, len(Hb))].reshape(-1, Hb.shape[-1])
    sample_R = Rb[:min(24, len(Rb))].reshape(-1, Rb.shape[-1])
    profile = hb_NoiseProfile(n_channels=int(len(hs)), hbo_scale=hs, hbr_scale=rs, phi_hbo=hb_safe_lag1(sample_H), phi_hbr=hb_safe_lag1(sample_R))
    return (profile, int(valid.sum()), int(len(valid)))


In [ ]:
from __future__ import annotations

def hb_external_synthetic_test(profile, dataset_name, pid):
    rng = np.random.default_rng(hb_stable_seed('RAW_D_SCREEN_EXTERNAL', dataset_name, pid, 'test'))
    trials = [hb_synthetic_trial(profile, rng, True) for _ in range(hb_EXTERNAL_TEST_POS)]
    trials.extend((hb_synthetic_trial(profile, rng, False) for _ in range(hb_EXTERNAL_TEST_NEG)))
    rng.shuffle(trials)
    out = dict(present=[], onset=[], peak=[], offset=[], censored=[], states=[], obs=[], state_feat=[])
    for tr in trials:
        ev = hb_evidence(tr['hbo'], tr['hbr'])
        out['present'].append(tr['present'])
        out['onset'].append(tr['onset'])
        out['peak'].append(tr['peak'])
        out['offset'].append(tr['offset'])
        out['censored'].append(tr['censored'])
        out['states'].append(tr['states'])
        out['obs'].append(hb_whole_vector(ev))
        out['state_feat'].append(hb_state_features(ev))
    q = {'present': np.asarray(out['present'], int), 'onset': np.asarray(out['onset'], float), 'peak': np.asarray(out['peak'], float), 'offset': np.asarray(out['offset'], float), 'censored': np.asarray(out['censored'], int), 'states': np.asarray(out['states'], int), 'obs': hb_finite_model_array(np.asarray(out['obs'], float)), 'state_feat': hb_finite_model_array(np.asarray(out['state_feat'], float))}
    return q

def hb__interp_epoch_channels(arr, source_t, target_t):
    arr = np.asarray(arr, float)
    out = np.full((arr.shape[0], len(target_t)), np.nan, float)
    for c in range(arr.shape[0]):
        x = arr[c]
        m = np.isfinite(x) & np.isfinite(source_t)
        if m.sum() < 2:
            continue
        out[c] = np.interp(target_t, source_t[m], x[m], left=x[m][0], right=x[m][-1])
    keep = np.mean(np.isfinite(out), axis=1) >= 0.95
    return out[keep]

def hb_real_external_predictors(store, pid):
    t, H, R, _ = hb__participant_arrays(store, pid)
    obs = []
    state = []
    kept = []
    for i in range(len(H)):
        h = hb__interp_epoch_channels(H[i], t, hb_FULL_T)
        r = hb__interp_epoch_channels(R[i], t, hb_FULL_T)
        C = min(len(h), len(r))
        if C < 2:
            continue
        ev = hb_evidence(h[:C], r[:C])
        obs.append(hb_whole_vector(ev))
        state.append(hb_state_features(ev))
        kept.append(i)
    if not obs:
        return (None, np.asarray([], int))
    return (dict(obs=hb_finite_model_array(np.asarray(obs, float)), state_feat=hb_finite_model_array(np.asarray(state, float))), np.asarray(kept, int))


In [ ]:
# Cell 7: Create the source-model namespaces
SOURCE_NS = _EngineProxy("source_")
HB_NS = _EngineProxy("hb_")


In [ ]:
# Cell 8: Load the unified 10-seed HemoBoundary source outputs
from __future__ import annotations

ALL_PIDS = tuple((f'VP{i:03d}' for i in range(1, 27)))

ACTIVE_RUN_SEED = 1

BASE_GLOBAL = dict(state_C=0.01, change_frac=0.3, global_alpha=10.0, response_threshold=0.5)

GLOBAL_TIMING_GRID = [dict(BASE_GLOBAL), dict(BASE_GLOBAL, state_C=0.003), dict(BASE_GLOBAL, state_C=0.03), dict(BASE_GLOBAL, change_frac=0.2), dict(BASE_GLOBAL, change_frac=0.4), dict(BASE_GLOBAL, global_alpha=3.0), dict(BASE_GLOBAL, global_alpha=30.0)]

GLOBAL_RESPONSE_GRID = GLOBAL_TIMING_GRID + [dict(BASE_GLOBAL, response_threshold=0.4), dict(BASE_GLOBAL, response_threshold=0.6)]

BASE_LBR = dict(radius=1.0, local_alpha=100.0, strength=0.5)

LBR_GRID = [dict(BASE_LBR), dict(BASE_LBR, radius=0.5), dict(BASE_LBR, radius=1.5), dict(BASE_LBR, local_alpha=30.0), dict(BASE_LBR, local_alpha=300.0), dict(BASE_LBR, strength=0.25), dict(BASE_LBR, strength=0.75)]

def base_stable_seed(*parts: Any) -> int:
    """Seed hash independent of the active robustness repeat."""
    raw = '|'.join((str(x) for x in parts)).encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2 ** 31 - 1)

def stable_seed(*parts: Any) -> int:
    """Seed 1 matches the uploaded script; seeds 2..10 add a repeat namespace."""
    if int(ACTIVE_RUN_SEED) == 1:
        return base_stable_seed(*parts)
    return base_stable_seed('MULTISEED_REPEAT', int(ACTIVE_RUN_SEED), *parts)

def canonical_cfg(cfg: Mapping[str, Any]) -> str:
    return json.dumps({k: cfg[k] for k in sorted(cfg)}, sort_keys=True, separators=(',', ':'))

def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_PART_NAME}.{os.getpid()}')
    compression = 'gzip' if str(path).endswith('.gz') else None
    df.to_csv(tmp, index=False, compression=compression)
    os.replace(tmp, path)

def atomic_json(obj: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_PART_NAME}.{os.getpid()}')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)

def try_mount_drive() -> None:
    if Path('/content/gdrive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/gdrive', force_remount=False)
    except Exception:
        pass

def add_pid_weight(cache):
    out = {}
    for pid, item in cache.items():
        out[pid] = {}
        for split, d0 in item.items():
            d = {k: np.asarray(v) for k, v in d0.items()}
            n = len(d['present'])
            d['pid'] = np.full(n, pid, dtype='U5')
            d['weight'] = np.ones(n, dtype=float)
            out[pid][split] = d
    return out

def training_rows(hb, caches, pids):
    return hb.cat([caches[p]['train'] for p in sorted(pids)])

def make_prediction_threshold(hb, raw, prob, threshold):
    raw = np.asarray(raw, float)
    prob = np.asarray(prob, float)
    on, pk, off = hb.enforce_geometry(raw[:, 0], raw[:, 1], raw[:, 2])
    miss = prob < float(threshold)
    on = on.astype(float)
    pk = pk.astype(float)
    off = off.astype(float)
    on[miss] = np.nan
    pk[miss] = np.nan
    off[miss] = np.nan
    return pd.DataFrame({'pred_response': (~miss).astype(int), 'pred_onset': on, 'pred_peak': pk, 'pred_offset': off})

def run_model_checks(source, hb):
    assert len(GLOBAL_TIMING_GRID) == 7
    assert len(GLOBAL_RESPONSE_GRID) == 9
    assert len(LBR_GRID) == 7
    assert len(GLOBAL_RESPONSE_GRID) * len(LBR_GRID) == 63
    assert BASE_GLOBAL == {'state_C': 0.01, 'change_frac': 0.3, 'global_alpha': 10.0, 'response_threshold': 0.5}
    assert BASE_LBR == {'radius': 1.0, 'local_alpha': 100.0, 'strength': 0.5}
    vals = [(1.0, i) for i in range(3)]
    vals.sort()
    assert vals[0][1] == 0
    print('HemoBoundary model checks: PASS')

def configure_repeat_seed(source, hb, seed: int) -> None:
    """
    Configure one complete robustness repeat.

    Seed 1 intentionally preserves the uploaded script's original deterministic
    behavior. Seeds 2..10 are independent repeat namespaces affecting:
      1) fresh RAW_D_SCREEN synthetic trial generation,
      2) inner participant-grouped folds,
      3) LBR participant-OOF folds,
      4) estimator random_state values.
    """
    global ACTIVE_RUN_SEED
    ACTIVE_RUN_SEED = int(seed)
    if not hasattr(source, '_MULTISEED_BASE_STABLE_SEED'):
        source._MULTISEED_BASE_STABLE_SEED = source.stable_seed
        source._MULTISEED_BASE_RANDOM_SEED = int(source.RANDOM_SEED)
    if not hasattr(hb, '_MULTISEED_BASE_STABLE_SEED'):
        hb._MULTISEED_BASE_STABLE_SEED = hb.stable_seed
        hb._MULTISEED_BASE_RANDOM_SEED = int(hb.RANDOM_SEED)
    source_base = source._MULTISEED_BASE_STABLE_SEED
    hb_base = hb._MULTISEED_BASE_STABLE_SEED
    if int(seed) == 1:
        source.stable_seed = source_base
        hb.stable_seed = hb_base
        source.RANDOM_SEED = int(source._MULTISEED_BASE_RANDOM_SEED)
        hb.RANDOM_SEED = int(hb._MULTISEED_BASE_RANDOM_SEED)
    else:

        def source_seed(*parts, _seed=int(seed), _base=source_base):
            return _base('MULTISEED_REPEAT', _seed, *parts)

        def hb_seed(*parts, _seed=int(seed), _base=hb_base):
            return _base('MULTISEED_REPEAT', _seed, *parts)
        source.stable_seed = source_seed
        hb.stable_seed = hb_seed
        source.RANDOM_SEED = int(source._MULTISEED_BASE_RANDOM_SEED) + int(seed) - 1
        hb.RANDOM_SEED = int(hb._MULTISEED_BASE_RANDOM_SEED) + int(seed) - 1
    np.random.seed(base_stable_seed('numpy_legacy', int(seed)) % (2 ** 32 - 1))

ROW_A = 'BASE_PLUS_A_TEMPORAL_BOUNDARY'

ROW_AB = 'BASE_PLUS_A_PLUS_B_RECOVERY'

ROW_ABC = 'BASE_PLUS_A_PLUS_B_PLUS_C_RESPONSE'

ROW_FULL = 'BASE_PLUS_A_PLUS_B_PLUS_C_PLUS_LBR'

LOCAL_OOF_FOLDS_COHERENT = 3

@dataclass
class CoherentStateModel:
    scaler: Any
    clf: Any

def fit_coherent_state(hb, data, state_C, audit, stage_prefix):
    audit.record(data, stage_prefix + '_state_encoder')
    F = hb.finite_model_array(data['state_feat']).astype(float, copy=False)
    X = F.reshape(-1, F.shape[-1])
    y = np.asarray(data['states'], int).reshape(-1)
    w = np.asarray(data['weight'], float)
    fw = np.repeat(w, F.shape[1])
    fw = fw / np.mean(fw)
    if len(X) > 60000:
        idx = np.random.default_rng(stable_seed('COHERENT_STATE_SUBSAMPLE', stage_prefix, *sorted(set(map(str, data['pid']))), float(state_C))).choice(len(X), 60000, replace=False)
        X, y, fw = (X[idx], y[idx], fw[idx])
    scaler = hb.StableScaler().fit(X, sample_weight=fw)
    clf = LogisticRegression(C=float(state_C), class_weight='balanced', max_iter=800, random_state=hb.RANDOM_SEED)
    clf.fit(scaler.transform(X), y, sample_weight=fw)
    return CoherentStateModel(scaler=scaler, clf=clf)

def coherent_state_probabilities(hb, model, x):
    hb.require_predictors_only(x)
    F = hb.finite_model_array(x['state_feat']).astype(float, copy=False)
    n, t, d = F.shape
    p0 = model.clf.predict_proba(model.scaler.transform(F.reshape(-1, d)))
    P = np.full((n * t, 5), 1e-08, float)
    for j, cls in enumerate(model.clf.classes_):
        P[:, int(cls)] = p0[:, j]
    P /= np.maximum(P.sum(axis=1, keepdims=True), 1e-12)
    return P.reshape(n, t, 5)

def coherent_state_features(hb, P):
    zero_response = np.zeros(len(P), float)
    full = hb.hard_state_features(P, zero_response)
    if full.shape[1] != 31:
        raise RuntimeError(f'Unexpected hard_state_features width {full.shape[1]} (expected 31).')
    return np.column_stack([full[:, :28], full[:, 29:]])

def coherent_ab_features(hb, state_model, x, change_frac):
    P = coherent_state_probabilities(hb, state_model, x)
    SX = coherent_state_features(hb, P)
    CX = hb.change_branch_features(x, float(change_frac))
    return (hb.finite_model_array(np.column_stack([SX, CX])), P)

@dataclass
class CoherentTimingHeads:
    scaler: Any
    regs: Dict[str, Any]
    names: Tuple[str, ...]

    def raw(self, X):
        Z = self.scaler.transform(X)
        return {name: self.regs[name].predict(Z) for name in self.names}

def fit_coherent_timing_heads(hb, X, data, alpha, names, audit, stage_prefix):
    audit.record(data, stage_prefix + '_timing_heads')
    names = tuple(names)
    w = np.asarray(data['weight'], float)
    scaler = hb.StableScaler().fit(X, sample_weight=w)
    Z = scaler.transform(X)
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    regs = {}
    for name in names:
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        if int(np.sum(mask)) < 15:
            raise RuntimeError(f'{stage_prefix}: too few rows for {name}.')
        rw = w[mask] / np.mean(w[mask])
        regs[name] = Ridge(alpha=float(alpha)).fit(Z[mask], y[mask], sample_weight=rw)
    return CoherentTimingHeads(scaler=scaler, regs=regs, names=names)

@dataclass
class CoherentABModel:
    hb: Any
    state: CoherentStateModel
    heads: CoherentTimingHeads
    change_frac: float

    def raw(self, x):
        X, P = coherent_ab_features(self.hb, self.state, x, self.change_frac)
        pred = self.heads.raw(X)
        on, pk, off = self.hb.enforce_geometry(np.asarray(pred['onset'], float), np.asarray(pred['peak'], float), np.asarray(pred['offset'], float))
        return (np.column_stack([on, pk, off]), P, X)

def fit_coherent_AB(hb, data, cfg, audit, stage_prefix='AB'):
    state = fit_coherent_state(hb, data, float(cfg['state_C']), audit, stage_prefix)
    X, _ = coherent_ab_features(hb, state, hb.predictor_view(data), float(cfg['change_frac']))
    heads = fit_coherent_timing_heads(hb, X, data, cfg['timing_alpha'], ('onset', 'peak', 'offset'), audit, stage_prefix)
    return CoherentABModel(hb=hb, state=state, heads=heads, change_frac=float(cfg['change_frac']))

@dataclass
class CoherentABCModel:
    hb: Any
    ab: CoherentABModel
    response_scaler: Any
    response_clf: Any
    response_threshold: float

    def raw(self, x):
        raw, P, X = self.ab.raw(x)
        prob = self.response_clf.predict_proba(self.response_scaler.transform(X))[:, 1]
        return (raw, np.asarray(prob, float), P)

    def predict(self, x):
        raw, prob, _ = self.raw(x)
        return make_prediction_threshold(self.hb, raw, prob, self.response_threshold)

def fit_coherent_ABC(hb, data, cfg, audit, stage_prefix='ABC'):
    ab = fit_coherent_AB(hb, data, cfg, audit, stage_prefix=stage_prefix + '_AB')
    X, _ = coherent_ab_features(hb, ab.state, hb.predictor_view(data), ab.change_frac)
    audit.record(data, stage_prefix + '_response_detector')
    w = np.asarray(data['weight'], float)
    y = np.asarray(data['present'], int)
    response_scaler = hb.StableScaler().fit(X, sample_weight=w)
    response_clf = LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=hb.RANDOM_SEED)
    response_clf.fit(response_scaler.transform(X), y, sample_weight=w / np.mean(w))
    return CoherentABCModel(hb=hb, ab=ab, response_scaler=response_scaler, response_clf=response_clf, response_threshold=float(cfg['response_threshold']))

def coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit_parent, context):
    pids = sorted(set(map(str, data['pid'])))
    n = len(data['pid'])
    raw = np.zeros((n, 3), float)
    rp = np.zeros(n, float)
    P = np.zeros((n, len(hb.GRID), 5), float)
    covered = np.zeros(n, int)
    folds = hb.grouped_folds(pids, stable_seed('COHERENT_LBR_OOF', context, canonical_cfg(abc_cfg), *pids), LOCAL_OOF_FOLDS_COHERENT)
    for held in folds:
        mask = np.isin(data['pid'], held)
        tr = hb.subset(data, ~mask)
        va = hb.subset(data, mask)
        tr_pids = sorted(set(map(str, tr['pid'])))
        audit = hb.FitAudit(tr_pids, set(outer_forbidden) | set(held), context + '/LBR_OOF')
        model = fit_coherent_ABC(hb, tr, abc_cfg, audit, stage_prefix='LBR_OOF_ABC')
        r, p, pp = model.raw(hb.predictor_view(va))
        raw[mask] = r
        rp[mask] = p
        P[mask] = pp
        covered[mask] += 1
        audit_parent.rows.extend(audit.rows)
    if not np.all(covered == 1):
        raise RuntimeError('Coherent LBR OOF anchor coverage failed.')
    return (raw, rp, P)

@dataclass
class CoherentFullModel:
    hb: Any
    abc: CoherentABCModel
    local_cfg: Dict[str, float]
    scalers: List[Any]
    regs: List[Any]

    def predict(self, x):
        raw, rp, P = self.abc.raw(x)
        profiles = self.hb.local_profiles(x, P, raw, rp, float(self.local_cfg['radius']))
        refined = np.asarray(raw, float).copy()
        for b in range(3):
            delta = self.regs[b].predict(self.scalers[b].transform(profiles[b]))
            radius = float(self.local_cfg['radius'])
            refined[:, b] += float(self.local_cfg['strength']) * np.clip(delta, -radius, radius)
        return make_prediction_threshold(self.hb, refined, rp, self.abc.response_threshold)

def fit_coherent_full(hb, data, abc_cfg, local_cfg, audit, outer_forbidden, context):
    abc = fit_coherent_ABC(hb, data, abc_cfg, audit, stage_prefix='FULL_ABC')
    raw_oof, rp_oof, P_oof = coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit, context)
    profiles = hb.local_profiles(hb.predictor_view(data), P_oof, raw_oof, rp_oof, float(local_cfg['radius']))
    audit.record(data, 'FULL_LBR_residual_heads_on_source_OOF_anchors')
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    scalers, regs = ([], [])
    for b, name in enumerate(('onset', 'peak', 'offset')):
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        scaler = hb.StableScaler().fit(profiles[b][mask])
        residual = y[mask] - raw_oof[mask, b]
        reg = Ridge(alpha=float(local_cfg['local_alpha'])).fit(scaler.transform(profiles[b][mask]), residual)
        scalers.append(scaler)
        regs.append(reg)
    return CoherentFullModel(hb=hb, abc=abc, local_cfg=dict(local_cfg), scalers=scalers, regs=regs)

RUN_PART_NAME = 'ALL_SEEDS_01_10'

EXTERNAL_ANALYSIS_NAME = 'HemoBoundary external validation'
OUTPUT_DIRNAME = 'analysis/External_Validation'

MAIN_SEED1_RESULT_DIRNAME = 'analysis/HemoBoundary_Main'

MAIN_RESULT_DIRNAME = 'analysis/HemoBoundary_Main'

MAIN_BANK_DIRNAME = 'analysis/HemoBoundary_Main'

EXT_MODEL_ABC = 'ABC'

EXT_MODEL_FULL = 'HEMOBOUNDARY'

EXT_MODEL_LABEL = {EXT_MODEL_ABC: 'Base + A + B + C', EXT_MODEL_FULL: 'HemoBoundary'}

EXT_ALL_SEEDS = tuple(range(1, 11))

EXT_BOOT_REPS = 20000

EXT_SIGNFLIP_REPS = 20000

EXT_TIMING = ('onset_mae_sec', 'peak_mae_sec', 'offset_mae_sec', 'duration_mae_sec')

EXT_METRICS = ('response_balanced_accuracy', *EXT_TIMING, 'selection_loss')

def ext_discover_shin_root(explicit: Optional[str]=None) -> Path:
    try_mount_drive()
    candidates: List[Path] = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.extend([Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/SHIN'), Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/SHIN')])
    for p in candidates:
        if p.is_dir():
            return p
    raise FileNotFoundError('Could not locate SHIN project root. Use --shin-root PATH.')

def ext_seed_result_root(shin_root: Path, seed: int) -> Path:
    p = shin_root / 'analysis/HemoBoundary_Main' / f'seed_{int(seed):02d}'
    if not p.is_dir():
        raise FileNotFoundError(f'Seed {int(seed):02d}: main-analysis result folder not found: {p}')
    return p

def ext_seed_cache_dir(shin_root: Path, seed: int) -> Path:
    p = shin_root / 'analysis/HemoBoundary_Main' / f'seed_{int(seed):02d}' / 'fresh_cache'
    if not (p.is_dir() and all(((p / f'{pid}.npz').is_file() for pid in ALL_PIDS))):
        raise FileNotFoundError(f'Seed {int(seed):02d}: complete source cache not found: {p}')
    return p

def ext_load_seed_caches(source, shin_root: Path, seed: int):
    cache_dir = ext_seed_cache_dir(shin_root, seed)
    caches0 = {pid: source.load_cache(cache_dir / f'{pid}.npz') for pid in ALL_PIDS}
    return (add_pid_weight(caches0), cache_dir)

def ext_seed_selected_json(shin_root: Path, seed: int, pid: str) -> Mapping[str, Any]:
    p = ext_seed_result_root(shin_root, seed) / 'checkpoints' / f'{pid}_SELECTED.json'
    if not p.is_file():
        raise FileNotFoundError(f'Seed {seed:02d} {pid}: selected config not found: {p}')
    return json.loads(p.read_text(encoding='utf-8'))

def ext_choose_modal_config(rows: pd.DataFrame, config_col: str, loss_col: str) -> Tuple[str, Dict[str, Any]]:
    """
    Rule:
      1. largest number of SHIN outer folds selecting exact config;
      2. lower mean saved SHIN inner loss;
      3. deterministic lexical tie-break.
    """
    summary = rows.groupby(config_col, dropna=False).agg(outer_folds_selected=('participant', 'nunique'), mean_inner_loss=(loss_col, 'mean')).reset_index().sort_values(['outer_folds_selected', 'mean_inner_loss', config_col], ascending=[False, True, True]).reset_index(drop=True)
    if len(summary) == 0:
        raise RuntimeError('No SHIN source configurations available for external freeze.')
    chosen = str(summary.iloc[0][config_col])
    audit = {'selection_rule': 'modal SHIN outer-selected configuration; tie-break by lower mean saved SHIN inner loss, then deterministic lexical order', 'modal_count': int(summary.iloc[0]['outer_folds_selected']), 'modal_fraction': float(summary.iloc[0]['outer_folds_selected'] / 26.0), 'modal_mean_inner_loss': float(summary.iloc[0]['mean_inner_loss']), 'all_counts': summary.to_dict(orient='records'), 'external_data_used_for_selection': False}
    return (chosen, audit)


In [ ]:
# Cell 9: Fit fixed source models and compute external participant-level statistics
from __future__ import annotations

def ext_seed_fixed_config(shin_root: Path, seed: int):
    """
    Fit the fixed source model for one robustness seed.

    IMPORTANT:
      - ABC global config is selected using SHIN outer selections only.
      - Full uses EXACTLY the same fixed ABC parent.
      - Only the LBR config is added for Full.
      - LBR is selected among SHIN folds whose Full global parent equals the
        chosen fixed ABC global parent, whenever possible.
    """
    records = []
    for pid in ALL_PIDS:
        selected = ext_seed_selected_json(shin_root, seed, pid)
        A = selected[ROW_A]['global']
        AB = selected[ROW_AB]['global']
        ABC = selected[ROW_ABC]['global']
        FULL = selected[ROW_FULL]['global']
        LBR = selected[ROW_FULL]['local']
        if not all((np.isclose(float(AB[key]), float(A[key])) for key in ('state_C', 'timing_alpha'))):
            raise RuntimeError(f'Seed {seed:02d} {pid}: A->B inheritance failed.')
        if not all((np.isclose(float(ABC[key]), float(AB[key])) for key in ('state_C', 'timing_alpha', 'change_frac'))):
            raise RuntimeError(f'Seed {seed:02d} {pid}: AB->C inheritance failed.')
        if canonical_cfg(ABC) != canonical_cfg(FULL):
            raise RuntimeError(f'Seed {seed:02d} {pid}: ABC parent changed in Full.')
        records.append({'participant': pid, 'ABC_cfg': canonical_cfg(ABC), 'ABC_inner_loss': float(selected[ROW_ABC]['inner_loss']), 'FULL_global_cfg': canonical_cfg(FULL), 'LBR_cfg': canonical_cfg(LBR), 'FULL_inner_loss': float(selected[ROW_FULL]['inner_loss'])})
    q = pd.DataFrame(records)
    abc_key, abc_audit = ext_choose_modal_config(q, 'ABC_cfg', 'ABC_inner_loss')
    abc_cfg = json.loads(abc_key)
    lbr_source = q[q['FULL_global_cfg'].astype(str) == str(abc_key)].copy()
    conditional_used = True
    if len(lbr_source) == 0:
        lbr_source = q.copy()
        conditional_used = False
    lbr_key, lbr_audit = ext_choose_modal_config(lbr_source, 'LBR_cfg', 'FULL_inner_loss')
    lbr_cfg = json.loads(lbr_key)
    audit = {'repeat_seed': int(seed), 'source_outer_folds': 26, 'hierarchical_inheritance_checked': True, 'ABC_parent_selection': abc_audit, 'LBR_selection': lbr_audit, 'LBR_selection_conditioned_on_fixed_ABC_parent': bool(conditional_used), 'LBR_selection_source_outer_folds': int(lbr_source['participant'].nunique()), 'fixed_ABC_cfg': abc_cfg, 'fixed_FULL_global_cfg': dict(abc_cfg), 'fixed_LBR_cfg': lbr_cfg, 'ABC_and_FULL_share_identical_global_parent': True, 'external_data_used_for_selection': False}
    return (abc_cfg, lbr_cfg, audit)

def ext_resolve_external_path(explicit: Optional[str], default_path: Path) -> Path:
    p = Path(explicit) if explicit else Path(default_path)
    candidates = [p]
    for q in candidates:
        if q.is_file():
            return q
    raise FileNotFoundError(p)

def ext_prepare_fixed_external_bank(hb, dataset_name: str, h5_path: Path):
    """
    Fixed target-physiology known-boundary bank.

    This is created ONCE before source-seed mutation, so all ten SHIN source
    seeds are evaluated on identical external known-boundary trials.
    """
    store = hb.load_external_hdf5(h5_path)
    pids = sorted(set(store['participant'].astype(str).tolist()))
    if len(pids) < 2:
        raise RuntimeError(f'{dataset_name}: expected multiple external participants.')
    banks = {}
    real_predictors = {}
    qc_rows = []
    for pid in pids:
        profile, valid_c, total_c = hb.noise_profile_from_external(store, pid)
        bank = hb.external_synthetic_test(profile, dataset_name, pid)
        banks[pid] = bank
        real_x, kept = hb.real_external_predictors(store, pid)
        real_predictors[pid] = (real_x, kept)
        qc_rows.append({'dataset': dataset_name, 'participant': pid, 'hdf5': str(h5_path), 'h5_schema': store['schema'], 'n_real_epochs': int(np.sum(store['participant'].astype(str) == str(pid))), 'usable_channels': int(valid_c), 'channels_before_validity_filter': int(total_c), 'phi_hbo': float(profile.phi_hbo), 'phi_hbr': float(profile.phi_hbr), 'median_hbo_noise_scale': float(np.median(profile.hbo_scale)), 'median_hbr_noise_scale': float(np.median(profile.hbr_scale)), 'fixed_external_bank_across_source_seeds': True})
    return {'store': store, 'pids': pids, 'banks': banks, 'real_predictors': real_predictors, 'qc': pd.DataFrame(qc_rows)}

def ext_fit_source_models(hb, caches, abc_cfg, lbr_cfg, seed: int):
    """
    Refit ABC and Full on ALL 26 SHIN TRAIN banks.

    Full shares the exact same fixed ABC parent as ABC and adds only LBR.
    """
    train = training_rows(hb, caches, ALL_PIDS)
    abc_audit = hb.FitAudit(ALL_PIDS, [], f'external_validation/seed={seed:02d}/ABC/all_SHIN_source')
    abc_model = fit_coherent_ABC(hb, train, abc_cfg, abc_audit, stage_prefix='EXTERNAL_EXT_ABC')
    full_audit = hb.FitAudit(ALL_PIDS, [], f'external_validation/seed={seed:02d}/FULL/all_SHIN_source')
    full_model = fit_coherent_full(hb, train, abc_cfg, lbr_cfg, full_audit, outer_forbidden=set(), context=f'external_validation/seed={seed:02d}/FULL/all_SHIN_source')
    audit = pd.DataFrame([{'repeat_seed': int(seed), 'external_model': EXT_MODEL_ABC, **row} for row in abc_audit.rows] + [{'repeat_seed': int(seed), 'external_model': EXT_MODEL_FULL, **row} for row in full_audit.rows])
    return (abc_model, full_model, audit)

def ext_predict_dataset_seed(hb, seed: int, dataset_name: str, info, abc_model, full_model, include_real_descriptive: bool):
    trial_parts = []
    pm_rows = []
    real_parts = []
    for pid in info['pids']:
        truth = info['banks'][pid]
        x = hb.predictor_view(truth)
        pred_abc = abc_model.predict(x)
        pred_full = full_model.predict(x)
        for model_name, pred in ((EXT_MODEL_ABC, pred_abc), (EXT_MODEL_FULL, pred_full)):
            m = hb.trial_metric_summary(truth, pred)
            pm_rows.append({'repeat_seed': int(seed), 'dataset': dataset_name, 'participant': pid, 'model': model_name, 'model_label': EXT_MODEL_LABEL[model_name], 'response_balanced_accuracy': float(m['response_balanced_accuracy']), 'onset_mae_sec': float(m['onset_mae_sec']), 'peak_mae_sec': float(m['peak_mae_sec']), 'offset_mae_sec': float(m['offset_mae_sec']), 'duration_mae_sec': float(m['duration_mae_sec']), 'selection_loss': float(m['selection_loss'])})
            trial_parts.append(pd.DataFrame({'repeat_seed': int(seed), 'dataset': dataset_name, 'participant': pid, 'model': model_name, 'model_label': EXT_MODEL_LABEL[model_name], 'synthetic_index': np.arange(len(pred), dtype=int), 'truth_present': np.asarray(truth['present'], int), 'truth_onset': np.asarray(truth['onset'], float), 'truth_peak': np.asarray(truth['peak'], float), 'truth_offset': np.asarray(truth['offset'], float), 'truth_censored': np.asarray(truth['censored'], int), 'pred_response': pred['pred_response'].to_numpy(int), 'pred_onset': pred['pred_onset'].to_numpy(float), 'pred_peak': pred['pred_peak'].to_numpy(float), 'pred_offset': pred['pred_offset'].to_numpy(float)}))
        if include_real_descriptive:
            real_x, kept = info['real_predictors'][pid]
            if real_x is not None and len(kept):
                for model_name, model in ((EXT_MODEL_ABC, abc_model), (EXT_MODEL_FULL, full_model)):
                    pred = model.predict(real_x)
                    real_parts.append(pd.DataFrame({'repeat_seed': int(seed), 'dataset': dataset_name, 'participant': pid, 'model': model_name, 'model_label': EXT_MODEL_LABEL[model_name], 'real_trial_index': kept, 'pred_response': pred['pred_response'].to_numpy(int), 'pred_onset': pred['pred_onset'].to_numpy(float), 'pred_peak': pred['pred_peak'].to_numpy(float), 'pred_offset': pred['pred_offset'].to_numpy(float), 'timing_truth_available': False, 'role': 'DESCRIPTIVE_ONLY'}))
    return (pd.concat(trial_parts, ignore_index=True), pd.DataFrame(pm_rows), pd.concat(real_parts, ignore_index=True) if real_parts else pd.DataFrame())

def ext_bootstrap_ci(x: Sequence[float], seed: int):
    a = np.asarray(x, float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return (np.nan, np.nan, np.nan)
    mean = float(np.mean(a))
    if len(a) == 1:
        return (mean, mean, mean)
    rng = np.random.default_rng(seed)
    vals = np.empty(EXT_BOOT_REPS, float)
    done = 0
    while done < EXT_BOOT_REPS:
        n = min(2000, EXT_BOOT_REPS - done)
        ix = rng.integers(0, len(a), size=(n, len(a)))
        vals[done:done + n] = np.mean(a[ix], axis=1)
        done += n
    lo, hi = np.quantile(vals, [0.025, 0.975])
    return (mean, float(lo), float(hi))

def ext_signflip_p(x: Sequence[float], seed: int):
    a = np.asarray(x, float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan
    obs = abs(float(np.mean(a)))
    if obs < 1e-15:
        return 1.0
    rng = np.random.default_rng(seed)
    hits = 0
    done = 0
    while done < EXT_SIGNFLIP_REPS:
        n = min(2000, EXT_SIGNFLIP_REPS - done)
        signs = rng.choice([-1.0, 1.0], size=(n, len(a)))
        stat = np.abs(np.mean(signs * a[None, :], axis=1))
        hits += int(np.sum(stat >= obs - 1e-15))
        done += n
    return float((hits + 1) / (EXT_SIGNFLIP_REPS + 1))

def ext_holm(pvals: Sequence[float]):
    p = np.asarray(pvals, float)
    out = np.full(len(p), np.nan, float)
    valid = np.flatnonzero(np.isfinite(p))
    if len(valid) == 0:
        return out
    order = valid[np.argsort(p[valid])]
    running = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        out[idx] = min(1.0, running)
    return out

def ext_per_seed_performance(pm: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (seed, dataset, model), q in pm.groupby(['repeat_seed', 'dataset', 'model'], sort=True):
        row = {'repeat_seed': int(seed), 'dataset': dataset, 'model': model, 'model_label': EXT_MODEL_LABEL[model], 'n_participants': int(q['participant'].nunique())}
        for metric in EXT_METRICS:
            row[metric] = float(np.mean(q[metric].to_numpy(float)))
        rows.append(row)
    return pd.DataFrame(rows)

def ext_summary(per_seed: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (dataset, model), q in per_seed.groupby(['dataset', 'model'], sort=True):
        row = {'dataset': dataset, 'model': model, 'model_label': EXT_MODEL_LABEL[model], 'n_source_seeds': int(q['repeat_seed'].nunique()), 'n_external_participants': int(q['n_participants'].iloc[0])}
        for metric in EXT_METRICS:
            x = q[metric].to_numpy(float)
            row[metric + '_mean'] = float(np.mean(x))
            row[metric + '_sd'] = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

def ext_paper_ready(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset in ('BALINT', 'KHAN'):
        for model in (EXT_MODEL_ABC, EXT_MODEL_FULL):
            q = summary[(summary['dataset'] == dataset) & (summary['model'] == model)]
            if len(q) != 1:
                continue
            q = q.iloc[0]
            rows.append({'dataset': dataset, 'model': EXT_MODEL_LABEL[model], 'n_external_participants': int(q['n_external_participants']), 'response_BA': f"{float(q['response_balanced_accuracy_mean']):.4f} ± {float(q['response_balanced_accuracy_sd']):.4f}", 'onset_MAE_s': f"{float(q['onset_mae_sec_mean']):.4f} ± {float(q['onset_mae_sec_sd']):.4f}", 'peak_MAE_s': f"{float(q['peak_mae_sec_mean']):.4f} ± {float(q['peak_mae_sec_sd']):.4f}", 'offset_MAE_s': f"{float(q['offset_mae_sec_mean']):.4f} ± {float(q['offset_mae_sec_sd']):.4f}", 'duration_MAE_s': f"{float(q['duration_mae_sec_mean']):.4f} ± {float(q['duration_mae_sec_sd']):.4f}"})
    return pd.DataFrame(rows)

def ext_primary_stats(pm: pd.DataFrame) -> pd.DataFrame:
    """
    Primary inference:
      within each external participant, average ABC-minus-Full timing gain
      (or Full-minus-ABC BA gain) across 10 source seeds;
      then infer across external participants.
    """
    gain_rows = []
    for (dataset, participant, seed), q in pm.groupby(['dataset', 'participant', 'repeat_seed'], sort=True):
        p = q.set_index('model')
        if not {EXT_MODEL_ABC, EXT_MODEL_FULL}.issubset(set(p.index)):
            raise RuntimeError(f'{dataset} {participant} seed {seed}: missing model.')
        for metric in EXT_METRICS:
            a = float(p.loc[EXT_MODEL_ABC, metric])
            b = float(p.loc[EXT_MODEL_FULL, metric])
            if metric == 'response_balanced_accuracy':
                gain = b - a
            else:
                gain = a - b
            gain_rows.append({'dataset': dataset, 'participant': participant, 'repeat_seed': int(seed), 'metric': metric, 'gain_positive_HemoBoundary_better': gain})
    gains = pd.DataFrame(gain_rows)
    participant_gain = gains.groupby(['dataset', 'participant', 'metric'], as_index=False)['gain_positive_HemoBoundary_better'].mean().rename(columns={'gain_positive_HemoBoundary_better': 'mean_gain_across_source_seeds'})
    seed_gain = gains.groupby(['dataset', 'repeat_seed', 'metric'], as_index=False)['gain_positive_HemoBoundary_better'].mean().rename(columns={'gain_positive_HemoBoundary_better': 'mean_gain_across_external_participants'})
    rows = []
    for dataset in sorted(pm['dataset'].unique()):
        for metric in EXT_METRICS:
            q = participant_gain[(participant_gain['dataset'] == dataset) & (participant_gain['metric'] == metric)]
            x = q['mean_gain_across_source_seeds'].to_numpy(float)
            sq = seed_gain[(seed_gain['dataset'] == dataset) & (seed_gain['metric'] == metric)]
            sx = sq['mean_gain_across_external_participants'].to_numpy(float)
            mean, lo, hi = ext_bootstrap_ci(x, stable_seed('OPTIONB_EXT_BOOT', dataset, metric))
            p = ext_signflip_p(x, stable_seed('OPTIONB_EXT_SIGN', dataset, metric))
            rows.append({'dataset': dataset, 'comparison': 'ABC -> HemoBoundary', 'metric': metric, 'n_external_participants': len(x), 'n_source_seeds': len(sx), 'mean_gain_positive_HemoBoundary_better': mean, 'ci95_low': lo, 'ci95_high': hi, 'participants_better': int(np.sum(x > 1e-12)), 'participants_equal': int(np.sum(np.isclose(x, 0.0, atol=1e-12))), 'participants_worse': int(np.sum(x < -1e-12)), 'source_seeds_positive': int(np.sum(sx > 1e-12)), 'source_seeds_equal': int(np.sum(np.isclose(sx, 0.0, atol=1e-12))), 'source_seeds_negative': int(np.sum(sx < -1e-12)), 'signflip_p': p})
    out = pd.DataFrame(rows)
    out['holm_p_4_timing_within_dataset'] = np.nan
    for dataset in out['dataset'].unique():
        ix = out.index[(out['dataset'] == dataset) & out['metric'].isin(EXT_TIMING)].tolist()
        out.loc[ix, 'holm_p_4_timing_within_dataset'] = ext_holm(out.loc[ix, 'signflip_p'].to_numpy(float))
    out['holm_p_global_external_timing'] = np.nan
    ix = out.index[out['metric'].isin(EXT_TIMING)].tolist()
    out.loc[ix, 'holm_p_global_external_timing'] = ext_holm(out.loc[ix, 'signflip_p'].to_numpy(float))
    return (gains, participant_gain, seed_gain, out)

def ext_run_source_seed(source, hb, shin_root: Path, out: Path, seed: int, external_sets, include_real_descriptive: bool, force: bool):
    cp = out / 'checkpoints'
    cp_trial = cp / f'seed_{seed:02d}_PER_TRIAL.csv.gz'
    cp_pm = cp / f'seed_{seed:02d}_PARTICIPANT_METRICS.csv'
    cp_real = cp / f'seed_{seed:02d}_REAL_DESCRIPTIVE.csv.gz'
    cp_fit = cp / f'seed_{seed:02d}_SOURCE_FIT_AUDIT.csv'
    cp_freeze = cp / f'seed_{seed:02d}_FREEZE_AUDIT.json'
    if cp_trial.is_file() and cp_pm.is_file() and cp_fit.is_file() and cp_freeze.is_file() and (not force):
        print(f'Seed {seed:02d}: RESUME', flush=True)
        return (pd.read_csv(cp_trial), pd.read_csv(cp_pm), pd.read_csv(cp_real) if cp_real.is_file() else pd.DataFrame())
    configure_repeat_seed(source, hb, int(seed))
    global COHERENT_SEED
    COHERENT_SEED = int(seed)
    caches, cache_dir = ext_load_seed_caches(source, shin_root, seed)
    abc_cfg, lbr_cfg, freeze_audit = ext_seed_fixed_config(shin_root, seed)
    print(f'Seed {seed:02d}: SHIN-only fixed ABC={abc_cfg}, LBR={lbr_cfg}', flush=True)
    abc_model, full_model, fit_audit = ext_fit_source_models(hb, caches, abc_cfg, lbr_cfg, seed)
    trial_parts = []
    pm_parts = []
    real_parts = []
    for dataset_name, info in external_sets.items():
        print(f'  {dataset_name}: without target adaptation', flush=True)
        trial, pm, real = ext_predict_dataset_seed(hb, seed, dataset_name, info, abc_model, full_model, include_real_descriptive)
        trial_parts.append(trial)
        pm_parts.append(pm)
        if len(real):
            real_parts.append(real)
    trial = pd.concat(trial_parts, ignore_index=True)
    pm = pd.concat(pm_parts, ignore_index=True)
    real = pd.concat(real_parts, ignore_index=True) if real_parts else pd.DataFrame()
    atomic_csv(trial, cp_trial)
    atomic_csv(pm, cp_pm)
    if len(real):
        atomic_csv(real, cp_real)
    atomic_csv(fit_audit, cp_fit)
    atomic_json({'analysis': EXTERNAL_ANALYSIS_NAME, 'repeat_seed': int(seed), 'source_cache': str(cache_dir), 'source_participants': list(ALL_PIDS), 'source_participant_count': 26, 'fixed_ABC_cfg': abc_cfg, 'fixed_FULL_global_cfg': dict(abc_cfg), 'fixed_LBR_cfg': lbr_cfg, 'ABC_and_FULL_share_identical_global_parent': True, 'freeze_selection_audit': freeze_audit, 'source_fit_uses_all_26_SHIN_train_banks': True, 'external_data_used_for_config_selection': False, 'external_data_used_for_source_fit': False, 'external_labels_used_for_tuning': False, 'external_adaptation': False, 'real_external_timing_truth_available': False, 'real_external_predictions_role': 'DESCRIPTIVE_ONLY', 'fixed_known_boundary_external_bank_across_source_seeds': True}, cp_freeze)
    return (trial, pm, real)

def run_external_adapter_checks():
    q = pd.DataFrame({'participant': ['A', 'B', 'C'], 'cfg': ['{"x":1}', '{"x":1}', '{"x":2}'], 'loss': [1.0, 2.0, 0.1]})
    chosen, audit = ext_choose_modal_config(q, 'cfg', 'loss')
    assert chosen == '{"x":1}'
    assert audit['modal_count'] == 2
    mean, lo, hi = ext_bootstrap_ci([0.1, 0.2, 0.3], 123)
    assert mean > 0
    assert lo > 0
    assert hi > 0
    print('External-validation adapter checks: PASS')


In [ ]:
# Cell 10: Merge all 10 source seeds and build the external-validation tables
from __future__ import annotations

def parse_external_validation_args():
    p = argparse.ArgumentParser()
    p.add_argument('--shin-root', default=None)
    p.add_argument('--output', default=None)
    p.add_argument('--balint-h5', default=None)
    p.add_argument('--khan-h5', default=None)
    p.add_argument('--datasets', default='BALINT,KHAN', help='BALINT, KHAN, or BALINT,KHAN')
    p.add_argument('--seeds', default='1,2,3,4,5,6,7,8,9,10')
    p.add_argument('--skip-real-descriptive', action='store_true')
    p.add_argument('--force', action='store_true')
    p.add_argument('--self-test-only', action='store_true')
    args, unknown = p.parse_known_args()
    if unknown:
        print('Ignoring notebook/runtime arguments:', unknown)
    return args

def parse_ext_seed_list(text: str) -> List[int]:
    vals = sorted(set((int(x.strip()) for x in str(text).split(',') if x.strip())))
    if not vals:
        raise ValueError('No source seeds requested.')
    if any((s not in EXT_ALL_SEEDS for s in vals)):
        raise ValueError(f'Source seeds must be 1..10, got {vals}.')
    return vals

def run_external_validation():
    args = parse_external_validation_args()
    with tempfile.TemporaryDirectory(prefix='hemoboundary_external_validation_') as td:
        td = Path(td)
        source = SOURCE_NS
        hb = HB_NS
        run_model_checks(source, hb)
        run_external_adapter_checks()
        if args.self_test_only:
            return
        shin_root = ext_discover_shin_root(args.shin_root)
        out = Path(args.output) if args.output else shin_root / OUTPUT_DIRNAME
        for sub in ('checkpoints', 'results', 'tables', 'audit'):
            (out / sub).mkdir(parents=True, exist_ok=True)
        requested_datasets = [x.strip().upper() for x in str(args.datasets).split(',') if x.strip()]
        if not requested_datasets or not set(requested_datasets).issubset({'BALINT', 'KHAN'}):
            raise ValueError('--datasets must contain BALINT and/or KHAN.')
        requested_seeds = parse_ext_seed_list(args.seeds)
        configure_repeat_seed(source, hb, 1)
        global COHERENT_SEED
        COHERENT_SEED = 1
        balint_path = ext_resolve_external_path(args.balint_h5, hb.BALINT_DEFAULT)
        khan_path = ext_resolve_external_path(args.khan_h5, hb.KHAN_DEFAULT)
        ext_paths = {'BALINT': balint_path, 'KHAN': khan_path}
        external_sets = {}
        print('=' * 180)
        print('HEMOBOUNDARY  — TABLE 3 — 10-SEED BÁLINT + KHAN WITHOUT TARGET ADAPTATION')
        print('=' * 180)
        print('SHIN root                       :', shin_root)
        print('Output                          :', out)
        print('Source robustness seeds         :', requested_seeds)
        print('External datasets               :', requested_datasets)
        print('Source participants / refit     : 26 SHIN')
        print('External config selection       : SHIN ONLY')
        print('ABC / Full global parent        : IDENTICAL')
        print('Full adds                       : LBR ONLY')
        print('External fitting                : NONE')
        print('External tuning                 : NONE')
        print('External adaptation             : NONE')
        print('Known-boundary external bank    : FIXED across all source seeds')
        print('Real external timing truth      : unavailable; descriptive only')
        for dataset_name in requested_datasets:
            print(f'Preparing fixed external bank: {dataset_name}', flush=True)
            external_sets[dataset_name] = ext_prepare_fixed_external_bank(hb, dataset_name, ext_paths[dataset_name])
        qc = pd.concat([external_sets[d]['qc'] for d in requested_datasets], ignore_index=True)
        atomic_csv(qc, out / 'audit' / 'EXTERNAL_FIXED_BANK_QC.csv')
        for seed in requested_seeds:
            ext_run_source_seed(source, hb, shin_root, out, int(seed), external_sets, include_real_descriptive=not args.skip_real_descriptive, force=bool(args.force))
        trial_parts = []
        pm_parts = []
        real_parts = []
        complete_seeds = []
        for seed in EXT_ALL_SEEDS:
            cp_trial = out / 'checkpoints' / f'seed_{seed:02d}_PER_TRIAL.csv.gz'
            cp_pm = out / 'checkpoints' / f'seed_{seed:02d}_PARTICIPANT_METRICS.csv'
            cp_real = out / 'checkpoints' / f'seed_{seed:02d}_REAL_DESCRIPTIVE.csv.gz'
            if cp_trial.is_file() and cp_pm.is_file():
                trial_parts.append(pd.read_csv(cp_trial))
                pm_parts.append(pd.read_csv(cp_pm))
                if cp_real.is_file():
                    real_parts.append(pd.read_csv(cp_real))
                complete_seeds.append(int(seed))
        if not pm_parts:
            raise RuntimeError('No completed external source-seed checkpoints found.')
        all_trial = pd.concat(trial_parts, ignore_index=True)
        all_pm = pd.concat(pm_parts, ignore_index=True)
        all_real = pd.concat(real_parts, ignore_index=True) if real_parts else pd.DataFrame()
        atomic_csv(all_trial, out / 'results' / 'TABLE3_EXTERNAL_KNOWN_BOUNDARY_PER_TRIAL_ALL_SEEDS.csv.gz')
        atomic_csv(all_pm, out / 'results' / 'TABLE3_EXTERNAL_PARTICIPANT_METRICS_ALL_SEEDS.csv.gz')
        if len(all_real):
            atomic_csv(all_real, out / 'results' / 'TABLE3_EXTERNAL_REAL_EXTERNAL_DESCRIPTIVE_ALL_SEEDS.csv.gz')
        if sorted(complete_seeds) != list(EXT_ALL_SEEDS):
            print('\n' + '=' * 180)
            print(' TABLE 3 PARTIAL RUN SAVED')
            print('=' * 180)
            print('Completed source seeds:', complete_seeds)
            atomic_json({'analysis': EXTERNAL_ANALYSIS_NAME, 'complete': False, 'completed_source_seeds': complete_seeds, 'required_source_seeds': list(EXT_ALL_SEEDS), 'external_tuning': False, 'external_adaptation': False}, out / 'audit' / 'TABLE3_EXTERNAL_RUN_MANIFEST.json')
            return
        for dataset_name in requested_datasets:
            expected_pids = set(external_sets[dataset_name]['pids'])
            q = all_pm[all_pm['dataset'] == dataset_name]
            for seed in EXT_ALL_SEEDS:
                for model in (EXT_MODEL_ABC, EXT_MODEL_FULL):
                    z = q[(q['repeat_seed'] == seed) & (q['model'] == model)]
                    if set(z['participant'].astype(str)) != expected_pids:
                        raise RuntimeError(f'{dataset_name} seed {seed:02d} {model}: participant coverage incomplete.')
        per_seed = ext_per_seed_performance(all_pm)
        summary = ext_summary(per_seed)
        paper = ext_paper_ready(summary)
        raw_gains, participant_gains, seed_gains, stats = ext_primary_stats(all_pm)
        atomic_csv(per_seed, out / 'results' / 'TABLE3_EXTERNAL_PER_SOURCE_SEED_PERFORMANCE.csv')
        atomic_csv(summary, out / 'tables' / 'Table3_EXTERNAL_10SEED_EXTERNAL_MEAN_SD.csv')
        atomic_csv(paper, out / 'tables' / 'Table3_EXTERNAL_10SEED_EXTERNAL_PAPER_READY.csv')
        atomic_csv(stats, out / 'tables' / 'Table3_ABC_VS_HEMOBOUNDARY_EXTERNAL_STATS.csv')
        atomic_csv(raw_gains, out / 'results' / 'External_Gains_Seed_Participant.csv.gz')
        atomic_csv(participant_gains, out / 'results' / 'External_Gains_Participant_Averaged_10Seeds.csv')
        atomic_csv(seed_gains, out / 'results' / 'External_Gains_Seed_Averaged_Participants.csv')
        freeze_audits = []
        for seed in EXT_ALL_SEEDS:
            p = out / 'checkpoints' / f'seed_{seed:02d}_FREEZE_AUDIT.json'
            freeze_audits.append(json.loads(p.read_text(encoding='utf-8')))
        atomic_json({'analysis': EXTERNAL_ANALYSIS_NAME, 'complete': True, 'source_seeds': list(EXT_ALL_SEEDS), 'source_participants': list(ALL_PIDS), 'source_participants_per_refit': 26, 'datasets': requested_datasets, 'external_paths': {d: str(ext_paths[d]) for d in requested_datasets}, 'fixed_known_boundary_bank_across_source_seeds': True, 'external_config_selection': 'SHIN-only modal hierarchical ABC parent per source seed; LBR modal selection conditioned on fixed ABC parent', 'ABC_and_FULL_share_identical_global_parent': True, 'Full_adds_only_LBR': True, 'external_data_used_for_model_selection': False, 'external_tuning': False, 'external_adaptation': False, 'external_labels_used_for_source_fit': False, 'external_labels_used_for_model_selection': False, 'ABC_external_model': 'refit once per source seed on all 26 SHIN TRAIN banks', 'HemoBoundary_external_model': 'same fixed ABC parent refit on all 26 SHIN TRAIN banks; source-participant OOF LBR residual fitting; add LBR only', 'real_external_timing_truth_available': False, 'real_external_predictions_role': 'DESCRIPTIVE_ONLY', 'primary_statistical_unit': 'external participant', 'source_seed_participant_rows_treated_as_independent': False, 'primary_inference': 'average ABC-to-Full gain across ten SHIN source seeds within each external participant, then bootstrap/sign-flip across external participants', 'holm_within_dataset_timing_family': 4, 'holm_global_external_timing_family': len(requested_datasets) * 4, 'seed_freeze_audits': freeze_audits}, out / 'audit' / 'TABLE3_EXTERNAL_RUN_MANIFEST.json')
        print('\n' + '=' * 180)
        print('External validation WITHOUT TARGET ADAPTATION EXTERNAL VALIDATION')
        print('=' * 180)
        print(paper.to_string(index=False))
        print('\n' + '=' * 180)
        print('TABLE 3 —  NUMERIC MEAN ± SD')
        print('=' * 180)
        print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
        print('\n' + '=' * 180)
        print('TABLE 3 —  ABC VS HEMOBOUNDARY EXTERNAL STATS')
        print('=' * 180)
        print(stats.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
        print('\nSUMMARY')
        print('  source robustness seeds         : 10 / 10')
        print('  SHIN source participants/refit  : 26 / 26')
        print('  external datasets               :', requested_datasets)
        print('  ABC / Full parent identical     : YES')
        print('  Full adds only LBR              : YES')
        print('  external model selection        : SHIN ONLY')
        print('  external fitting/tuning         : NO')
        print('  external adaptation             : NO')
        print('  external labels in source fit   : NO')
        print('  fixed ext bank across src seeds : YES')
        print('  real timing accuracy claim      : No')
        print('  inference unit                  : external participant')
        print('  pseudo-replication across seeds : No')
        print('  saved                           :', out)


In [ ]:
# Cell 11: Run the 10-seed Bálint + Khan external validation
run_external_validation()
